# ZelusBench: Attention Updating — LightPins transform_prob=0.1 while randomizing everything else.Higher probability = more transforms to propagate.- 15 scenarios

In [ ]:
import kaggle_benchmarks as kbenchimport numpy as npimport randomimport timefrom zelusbench.scenarios.config import ScenarioConfig, QueryTypefrom zelusbench.scenarios.generator import ScenarioGeneratorfrom zelusbench.evaluation.scorer import score_query, score_responseTRANSFORM_PROB = 0.1SEEDS = 15print(f"ZelusBench Attention Updating — Light (transform_prob={TRANSFORM_PROB})")

In [ ]:
@kbench.task(name="zelusbench_attn_updating_light")def zelusbench_attn_updating_light(llm) -> tuple[float, float]:    _start = time.time()    print(f"Running {SEEDS} scenarios...")    print("=" * 60)    all_scores = []    for i in range(SEEDS):        bg_rng = random.Random(i * 7919)        cfg = ScenarioConfig.randomize_except(bg_rng, pinned={            "transform_prob": TRANSFORM_PROB,            "transform_types": ["rotation", "translation"],            "num_queries": 3,            "seed": i,        })        scenario = ScenarioGenerator(cfg).generate(f"updating_{i}")        response = llm.prompt(scenario.prompt)        scores = score_response(response, scenario)        all_scores.extend(scores)        avg = float(np.mean([s.score for s in scores]))        bg = f"dim={cfg.dim} lb={cfg.leaf_bias} pts={cfg.num_points}"        print(f"  [{i+1}/{SEEDS}] avg={avg:.2f} | {bg}")    overall = float(np.mean([s.score for s in all_scores]))    std = float(np.std([s.score for s in all_scores]))    print(f"\nOverall: {overall:.3f} +/- {std:.3f} | {len(all_scores)} queries | {time.time()-_start:.1f}s")    kbench.assertions.assert_true(overall >= 0, expectation=f"Valid answers (got {overall:.3f})")    return overall, stdzelusbench_attn_updating_light.run(llm=kbench.llm)

In [ ]:
%choose zelusbench_attn_updating_light